# 03 — Feature Engineering

**Goal:** turn the cleaned data into a model-ready table — time features, distance features, encodings.

Input: `data/processed/dataset_clean.csv`
Output: `data/processed/dataset_model_ready.csv`


## 1. Setup

In [34]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

CLEAN_PATH = Path("../data/dataset_clean.csv")
MODEL_READY_PATH = Path("../data/dataset_model_ready.csv")
MODELS_DIR = Path("../model")

TARGET = "Delivery_Duration_Minutes"

df = pd.read_csv(CLEAN_PATH)
print(df.shape)
df.head()

(85199, 23)


,Order_ID,User_ID,Restaurant_ID,Driver_ID,Item_Name,Quantity,Total_Price,Order_Time,Delivery_Time,Delivery_Duration_Minutes,...,Driver_Vehicle,Restaurant_Lat,Restaurant_Lon,Customer_Lat,Customer_Lon,Driver_Lat,Driver_Lon,Delivery_Distance_km,Traffic_Level,Driver_Availability
0,1,U3522,358,485,Fried Chicken,3,273.72,2025-06-16 08:32:00,2025-06-16 09:11:00,31.0,...,Motorbike,31.195082,29.921931,31.191404,29.904982,31.215658,29.910664,1.666106,High,Offline
1,2,U9214,316,65,Sandwich,3,365.82,2025-06-03 21:27:00,2025-06-03 22:00:00,8.0,...,Motorbike,30.605729,31.503079,30.586047,31.485820,30.580329,31.502380,2.738698,Low,Online
2,4,U3612,420,32,Sushi,2,221.18,2025-06-13 02:30:00,2025-06-13 03:22:00,17.0,...,Car,31.041846,31.381229,31.035773,31.380440,31.054690,31.401187,0.677498,Low,Online
3,5,U3492,73,364,Koshary,5,355.55,2025-06-06 09:48:00,2025-06-06 10:32:00,8.0,...,Motorbike,31.024141,31.376104,31.026023,31.396881,31.035350,31.389315,1.994769,High,Online
4,6,U7439,750,270,Sushi,3,205.44,2025-06-04 12:16:00,2025-06-04 12:45:00,23.0,...,Bicycle,31.024140,31.387817,31.029004,31.373869,31.022573,31.380646,1.436807,Medium,Online


## 2. Time features

`Delivery_Time` is dropped later — it only exists *after* the order is delivered, so using it as a feature would leak the outcome. `Order_Time` is fine, because it's known the moment the order is placed.

In [35]:
df["Order_Time"] = pd.to_datetime(df["Order_Time"], dayfirst=True, errors="coerce")

df["order_hour"] = df["Order_Time"].dt.hour
df["order_dayofweek"] = df["Order_Time"].dt.dayofweek          # 0=Mon
df["order_is_weekend"] = df["order_dayofweek"].isin([4, 5]).astype(int)  # Fri/Sat weekend in Egypt
df["order_is_peak"] = df["order_hour"].isin([12, 13, 19, 20, 21, 22]).astype(int)
df["order_part_of_day"] = pd.cut(
    df["order_hour"], bins=[-1, 5, 11, 16, 20, 23],
    labels=["Late_Night", "Morning", "Afternoon", "Evening", "Night"]
)

df[["order_hour", "order_dayofweek", "order_is_weekend", "order_is_peak", "order_part_of_day"]].head()

C:\Users\Youssef\AppData\Local\Temp\ipykernel_29200\3207629523.py:1: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["Order_Time"] = pd.to_datetime(df["Order_Time"], dayfirst=True, errors="coerce")


,order_hour,order_dayofweek,order_is_weekend,order_is_peak,order_part_of_day
0,8,0,0,0,Morning
1,21,1,0,1,Night
2,2,4,1,0,Late_Night
3,9,4,1,0,Morning
4,12,2,0,1,Afternoon


## 3. Geo / distance features

We recompute the straight-line (Haversine) distance ourselves rather than trusting the dataset's `Delivery_Distance_km` alone — it's a useful independent feature, and also lets us cross-check the stated distance elsewhere.

In [36]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

df["haversine_restaurant_customer_km"] = haversine(
    df["Restaurant_Lat"], df["Restaurant_Lon"], df["Customer_Lat"], df["Customer_Lon"])
df["haversine_driver_restaurant_km"] = haversine(
    df["Driver_Lat"], df["Driver_Lon"], df["Restaurant_Lat"], df["Restaurant_Lon"])

df[["Delivery_Distance_km", "haversine_restaurant_customer_km", "haversine_driver_restaurant_km"]].describe().T

,count,mean,std,min,25%,50%,75%,max
Delivery_Distance_km,85199.0,2.166098,1.038995,0.008839,1.356168,2.124032,2.925337,5.277892
haversine_restaurant_customer_km,85199.0,2.167908,1.040415,0.008829,1.357311,2.125444,2.927625,5.603397
haversine_driver_restaurant_km,85199.0,2.163426,1.036979,0.008630,1.353144,2.118138,2.918918,5.607220


## 4. Price feature

In [37]:
df["unit_price"] = df["Total_Price"] / df["Quantity"].replace(0, np.nan)
df[["Total_Price", "Quantity", "unit_price", "Item_Name"]].head()

,Total_Price,Quantity,unit_price,Item_Name
0,273.72,3,91.24,Fried Chicken
1,365.82,3,121.94,Sandwich
2,221.18,2,110.59,Sushi
3,355.55,5,71.11,Koshary
4,205.44,3,68.48,Sushi


## 5. Interaction feature: traffic × distance

Traffic matters more on longer routes than short ones — a multiplicative interaction term captures that better than the two features separately. This turns out to be the single strongest predictor (see notebook 4).

In [38]:
TRAFFIC_ORDINAL = {"Low": 1, "Medium": 2, "High": 3}
traffic_level_ord = df["Traffic_Level"].map(TRAFFIC_ORDINAL)
df["traffic_x_distance"] = traffic_level_ord * df["Delivery_Distance_km"]
df[["Traffic_Level", "Delivery_Distance_km", "traffic_x_distance"]].head()

,Traffic_Level,Delivery_Distance_km,traffic_x_distance
0,High,1.666106,4.998319
1,Low,2.738698,2.738698
2,Low,0.677498,0.677498
3,High,1.994769,5.984306
4,Medium,1.436807,2.873614


## 6. One-hot encode low-cardinality categoricals

In [39]:
onehot_cols = ["Item_Name", "Traffic_Level", "Driver_Vehicle", "Payment_Method",
               "Driver_Availability", "order_part_of_day", "City"]
df = pd.get_dummies(df, columns=onehot_cols, drop_first=True)
print(df.shape)

(85199, 50)


## 7. Standarize the column names

In [40]:
df.columns = df.columns.str.strip().str.lower()
df.columns.to_list()

['order_id',
 'user_id',
 'restaurant_id',
 'driver_id',
 'quantity',
 'total_price',
 'order_time',
 'delivery_time',
 'delivery_duration_minutes',
 'order_status',
 'restaurant_lat',
 'restaurant_lon',
 'customer_lat',
 'customer_lon',
 'driver_lat',
 'driver_lon',
 'delivery_distance_km',
 'order_hour',
 'order_dayofweek',
 'order_is_weekend',
 'order_is_peak',
 'haversine_restaurant_customer_km',
 'haversine_driver_restaurant_km',
 'unit_price',
 'traffic_x_distance',
 'item_name_fried chicken',
 'item_name_koshary',
 'item_name_pasta',
 'item_name_pizza',
 'item_name_salad',
 'item_name_sandwich',
 'item_name_shawarma',
 'item_name_sushi',
 'traffic_level_low',
 'traffic_level_medium',
 'driver_vehicle_car',
 'driver_vehicle_motorbike',
 'payment_method_credit card',
 'payment_method_wallet',
 'driver_availability_online',
 'order_part_of_day_morning',
 'order_part_of_day_afternoon',
 'order_part_of_day_evening',
 'order_part_of_day_night',
 'city_assiut',
 'city_cairo',
 'city_gi

## 7. Drop leakage / redundant / pure-identifier columns

- `delivery_time` — only exists after the outcome is known
- `order_time` — already decomposed into hour/day/peak
- `Order_ID`, `User_ID`, `restaurant_id`, `driver_id`— pure identifiers, no signal
- `order_status` — constant now (all `Delivered`) after filtering in notebook 2

In [41]:
drop_cols = ["delivery_time", "order_time", "order_id" ,"user_id" ,"restaurant_id" ,"driver_id", "order_status"]
df_final = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

before = len(df_final)
df_final = df_final.dropna()
print(f"Dropped {before - len(df_final)} rows with residual NaNs")
print("Final shape:", df_final.shape)

Dropped 0 rows with residual NaNs
Final shape: (85199, 43)


## 10. Save model-ready data + feature manifest

In [ ]:
df_final.to_csv(MODEL_READY_PATH, index=False)

feature_columns = [c for c in df_final.columns if c != TARGET.lower()]
with open(MODELS_DIR / "feature_manifest.json", "w") as f:
    json.dump({"n_features": len(feature_columns), "feature_columns": feature_columns}, f, indent=2)

print(f"Saved -> {MODEL_READY_PATH}  shape={df_final.shape}")
print(f"Saved -> {MODELS_DIR / 'feature_manifest.json'}")

Saved -> ..\data\dataset_model_ready.csv  shape=(85199, 43)
Saved -> ..\model\feature_manifest.json
